> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **SAS Evaluation**
This notebook measures *Confirmation Bias* using the `stsb-roberta-large` model to weigh the semantic similarity between sentences.

In [1]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.sas import compute_sas_metrics

## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [2]:
# Define the path to the generated results (JSONL format)
# file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
file_path = "../data/interim/5_mmlu_pro_gpt_4o_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

Loaded 2 samples.


## **Evaluation**
Compute the evaluation metrics by sending the generated results to the model, that calculates the Separation margin and the CB_SAS score.

In [3]:
# Compute the SAS metrics
df_evaluated = compute_sas_metrics(df_results, tau_sep=0.03)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [4]:
# Aggregation of the overall Confirmation Bias mean
mean_cb = float(df_evaluated["CB_SAS"].mean())

# Aggregation based on the Reliability Gate generated (Sep >= 0.03)
reliable_mean_cb = float(df_evaluated[df_evaluated["sas_reliable"] == True]["CB_SAS"].mean())
reliable_rate = float(df_evaluated["sas_reliable"].mean())

print("Results of the SAS Evaluation:")
print(f"Mean Confirmation Bias (All samples): {mean_cb:.6f}")
print(f"Mean Confirmation Bias (Reliable samples): {reliable_mean_cb:.6f}")
print(f"Reliability Rate (% of Sep >= TAU):  {reliable_rate:.2%}")

display(df_evaluated[["sample", "model", "Sep", "CB_SAS", "CB_SAS_clipped", "sas_reliable"]].head())

Results of the SAS Evaluation:
Mean Confirmation Bias (All samples): -0.029725
Mean Confirmation Bias (Reliable samples): nan
Reliability Rate (% of Sep >= TAU):  0.00%


,sample,model,Sep,CB_SAS,CB_SAS_clipped,sas_reliable
0,1,gpt-4o,0.0,0.075911,0.075911,False
1,2,gpt-4o,0.0,-0.135361,-0.135361,False


## **Exporting**
Export the results to a CSV file for later use in the final analysis notebook.

In [5]:
# Extract filename from the input path
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
output_file = f"../data/interim/{base_name}_sas.csv"

# Save the evaluated DataFrame to the interim data directory
df_evaluated.to_csv(output_file, index=False)
print(f"Saved evaluation file to {output_file}")

Saved evaluation file to ../data/interim/5_mmlu_pro_gpt_4o_sas.csv
